In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# Jupyter Notebook Configuration

This section configures Jupyter to display all outputs in a cell, not just the last one. This is useful for debugging and seeing intermediate results.

#### Lets try LlamaIndex first

In [3]:
!uv pip install llama-index

Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 1 package in 45ms


# Install LlamaIndex

Install the LlamaIndex library, which is used for building retrieval-augmented generation (RAG) systems. This library provides tools for indexing and querying documents.

# Install LlamaIndex

Install the LlamaIndex library, which is used for building retrieval-augmented generation (RAG) systems. This library provides tools for indexing and querying documents.

In [4]:
!uv pip install llama-index-embeddings-huggingface
!uv pip install llama-index-llms-lmstudio

Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 1 package in 17ms
Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 1 package in 10ms


# Install Additional Dependencies

Install additional dependencies for LlamaIndex:
- **HuggingFace Embeddings**: For creating vector embeddings from text.
- **LMStudio Integration**: For connecting to a local LLM server.

In [5]:
import os
import openai
from dotenv import load_dotenv
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.vector_stores.types import VectorStoreQueryMode
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.llms.lmstudio import LMStudio

load_dotenv()

W0717 16:59:11.955000 347 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


True

# Import Required Libraries

Import all necessary libraries for building the RAG system. This includes:
- **LlamaIndex Core**: For indexing and querying documents.
- **HuggingFace Embeddings**: For creating vector embeddings.
- **LMStudio**: For integrating with a local LLM server.
- **Environment Variables**: For managing API keys and configurations.

In [6]:

# Configure the client to use LM Studio's local server
client = openai.OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"  # LM Studio doesn't require a real API key
)


# Set up LM Studio LLM
llm = LMStudio(
    base_url="http://localhost:1234/v1",
    model_name="mistral-7b-instruct-v0.3",  # Specify the model name
    temperature=0.2
)

# langchain openai





# Configure LM Studio

Set up the LM Studio client and LLM integration. This includes:
- **OpenAI-Compatible Client**: For testing connectivity.
- **LMStudio LLM**: For generating responses using a local model.

In [7]:
# Use chat completions
response = client.chat.completions.create(
    model="mistral-7b-instruct-v0.3",  # or whatever model name LM Studio shows
    messages=[
        {"role": "user", "content": "Hello, how are you? I am testing the LM Studio."},
    ],
    temperature=0.9,
    max_tokens=150,

)


# Test LM Studio Connection

Send a test query to LM Studio using the OpenAI-compatible API. This ensures the server is running and responding correctly.

In [8]:
print("Response from LM Studio:")
# Process the stream
print(response.choices[0].message.content)
print()  # New line at the end

Response from LM Studio:
 Hello there! I'm doing well and happy to assist you with questions or tasks related to the L Märтво Sohn studio. How can I help? Let me guide you through our collaboration as seamlessly as possible.



# Display Test Response

Print the response from LM Studio to verify that the connection and query are working as expected.

In [9]:
llm.complete("Can you tell me what is the capital of UAE?")

CompletionResponse(text=' The capital city of United Arab Emirates (UAE) is not a clearly defined entity, as there isn\'t one specific national capital as one might find in other countries. Each of its seven emírates maintains autonomous power under the loose agreement established with federation founding on December 2,1971, although Abu Dhabi has the dominant status due to its large oil resources and strategic geographic location. The federal government resides at an city called\'The political capital\', called "al-ī dīʾiyah al-īammah" in Arabic, which is located within the territory of Abu Dhabi. However, for practical purposes, Dubai, within the Emirate of Dubai, is considered more of a cultural and economic hub.', additional_kwargs={'id': 'chatcmpl-y43fhwl4imvzf08b270js', 'object': 'chat.completion', 'created': 1752742754, 'model': 'mistral-7b-instruct-v0.3', 'usage': {'prompt_tokens': 17, 'completion_tokens': 163, 'total_tokens': 180}, 'stats': {}, 'system_fingerprint': 'mistral-

In [10]:
# let's now download a file
!curl -o data/paul_graham_essay.txt https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 75042  100 75042    0     0   229k      0 --:--:-- --:--:-- --:--:--  229k


In [11]:
documents = SimpleDirectoryReader("data").load_data()
print(f"Number of documents loaded: {len(documents)}")

Number of documents loaded: 1


In [12]:
# If you want to use text splitting, you can do it like this:
# chunk using llama_index
text_splitter = SimpleNodeParser.from_defaults(
    chunk_size=800,  # Adjust chunk size as needed
    chunk_overlap=100  # Adjust overlap as needed
)
chunks = text_splitter.get_nodes_from_documents(documents)
print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 26


In [13]:
chunks[0].text[:500]  # Display the first 500 characters of the first chunk

'What I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn\'t write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.\n\nThe first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing'

In [14]:
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")  # Set the embed model globally
Settings.llm = llm  # Set the LLM client globally

In [15]:
index = VectorStoreIndex.from_documents(
    documents,
    # we can optionally override the embed_model here
    embed_model=Settings.embed_model,
    show_progress=True
)

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/22 [00:00<?, ?it/s]

In [16]:
index.storage_context.persist(persist_dir="storage")

In [17]:
# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=3,
    vector_store_query_mode=VectorStoreQueryMode.DEFAULT
)

In [18]:
# configure response synthesizer
response_synthesizer = get_response_synthesizer()

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.2)],
)

In [19]:
question = "What did Robert Morris do?"

In [20]:
# response_synthesizer.get_prompts()
# query
retrieved_examples = query_engine.retrieve(question)

print("Retrieved examples:")
for example in retrieved_examples:
    print(f"- {example.text[:200]}...")  # Print the first 200 characters of each retrieved example

input_context = "\n".join([example.text for example in retrieved_examples])

Retrieved examples:
- After I moved to New York I became her de facto studio assistant.

She liked to paint on big, square canvases, 4 to 5 feet on a side. One day in late 1994 as I was stretching one of these monsters the...
- The students and faculty in the painting department at the Accademia were the nicest people you could imagine, but they had long since arrived at an arrangement whereby the students wouldn't require t...


In [21]:
input_template =  """
    You are given a context from documents. Use this context to answer a question. Be consise and to the point.

    Context:
    {context}

    Question: {question}

    if the context does not contain enough information to answer the question, say "I don't know based on the given context". Do not make up answers.
    """


# Format a prompt:

def create_prompt(question, context):
    prompt = input_template.format(question=question, context=context)
    return prompt

final_prompt = create_prompt(question, context=input_context)

print("Question:")
print(question)
# print(final_prompt)

Question:
What did Robert Morris do?


In [22]:
print(llm.complete(final_prompt).text)

 Robert Morris, according to the given context, studied graduate school at Harvard. He also helped his project partner by writing some software, such as a shopping cart and an image resizing software for their web-based venture. Specific details about what exactly he worked on in school or other professional positions aren't provided in this context.


#### Let's try Langchain now

In [23]:
! uv pip install langchain langchain-community chromadb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 3 packages in 28ms


In [24]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader, DirectoryLoader

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

docs = DirectoryLoader(
    "data",
    glob="**/*.txt",
    loader_cls=TextLoader
).load()

print(f"Number of documents loaded: {len(docs)}")


chunks = text_splitter.split_documents(docs)

print(f"Number of chunks created: {len(chunks)}")


Number of documents loaded: 1
Number of chunks created: 136


In [25]:
! uv pip install langchain-huggingface

Using Python 3.12.2 environment at: /Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env
Audited 1 package in 7ms


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

encode_kwargs = {'normalize_embeddings': True}

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2",
                                         encode_kwargs=encode_kwargs)



vectorstore = Chroma.from_documents(collection_name="doc-index", documents=chunks, 
                                    embedding=embedding_model)


In [27]:
# lets do a similarity search

print("Performing similarity search... with question:", question)

vectorstore.similarity_search(question, k=3)

# looks good, let's try with a different question

vectorstore.similarity_search(
        "What is the main idea of the essay?" , k=5, )

Performing similarity search... with question: What did Robert Morris do?


[Document(metadata={'source': 'data/paul_graham_essay.txt'}, page_content='One day in 2010, when he was visiting California for interviews, Robert Morris did something astonishing: he offered me unsolicited advice. I can only remember him doing that once before. One day at Viaweb, when I was bent over double from a kidney stone, he suggested that it would be a good idea for him to take me to the hospital. That was what it took for Rtm to offer unsolicited advice. So I remember his exact words very clearly. "You know," he said, "you should make sure Y Combinator isn\'t the last cool thing you do."'),
 Document(metadata={'source': 'data/paul_graham_essay.txt'}, page_content='In September, Robert rebelled. "We\'ve been working on this for a month," he said, "and it\'s still not done." This is funny in retrospect, because he would still be working on it almost 3 years later. But I decided it might be prudent to recruit more programmers, and I asked Robert who else in grad school with him w

[Document(metadata={'source': 'data/paul_graham_essay.txt'}, page_content='something people usually take for granted, just as you can after days of trying to write an essay about something people usually take for granted.'),
 Document(metadata={'source': 'data/paul_graham_essay.txt'}, page_content="Now that I could write essays again, I wrote a bunch about topics I'd had stacked up. I kept writing essays through 2020, but I also started to think about other things I could work on. How should I choose what to do? Well, how had I chosen what to work on in the past? I wrote an essay for myself to answer that question, and I was surprised how long and messy the answer turned out to be. If this surprised me, who'd lived it, then I thought perhaps it would be interesting to other people, and encouraging to those with similarly messy lives. So I wrote a more detailed version for others to read, and this is the last sentence of it.\n\n\n\n\n\n\n\n\n\nNotes"),
 Document(metadata={'source': 'dat

In [28]:
retriever = vectorstore.as_retriever(search_type='mmr')

In [29]:
# let's create the langchain chain
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI as langchain_OpenAI
from langchain.prompts import PromptTemplate

# use langchain openai 
# Configure the client to use LM Studio's local server
client = langchain_OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"  # LM Studio doesn't require a real API key
)

llama_prompt = PromptTemplate(
       template=input_template, input_variables=["context", "question"]
   )

chain_type_kwargs = {"prompt": llama_prompt}

                                               
chain = RetrievalQA.from_chain_type(
    llm=client,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs,
    verbose=True
)

/var/folders/gb/2bddxvts0jj8qlj35_f1x9bc0000gp/T/ipykernel_347/1997148820.py:8: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  client = langchain_OpenAI(


In [30]:
response = chain.invoke("Can we compare Viaweb and Y Combinator? What are the differences?")  # Example question



> Entering new RetrievalQA chain...

> Finished chain.


In [31]:
print(response['result'])

 Based on the given context, we can identify that Viaweb and Y Combinator are different entities. Viaweb is a company started by Paul Graham in the 90s, focusing on providing application services through the web. On the other hand, Y Comcombinator is an organisation started by Paul Graham which aims to help startups grow their businesses using funding and resources. The differences between the two are: Via web is a service provider while Y Combinator is an incubator or startup program, Via web was started in the 90s and Y Combinator was started after ViaWeb.
